## 📋 Quick Reference: All QX Computation Versions

### **CPU Baseline**
- **Function**: `compute_Qx_cpu()`
- **Performance**: 1x (baseline)
- **Use Case**: Reference implementation, small datasets
- **Pro**: Simple, no GPU needed
- **Con**: Slow for large datasets

---

### **Version 1: Sequential GPU Adaptive** ⭐ **RECOMMENDED FOR small datasets or for debugging phase**
- **Function**: `compute_Qx_sequential_gpu_pure_adaptive()`
- **Strategy**: 
  - Processes tests **one-by-one**
  - Each test gets **perfectly-sized matrix** (n_snps × n_perm)
  - **Adaptive OOM handling**: automatically excludes tests too large for GPU
  - **Zero memory waste**: small tests don't pay penalty for large ones
- **Pro**: 
  - ✅ Most memory-efficient
  - ✅ Handles mixed test sizes perfectly
  - ✅ Automatic OOM recovery
  - ✅ Simple and reliable
- **Con**: Not as fast as batch for uniform test sizes

---

### **Version 2: Batch GPU with Per-Test Matrices** 🚀 **FASTEST**
- **Function**: `compute_Qx_batch_gpu_adaptive()`
- **Strategy**:
  - Processes tests in **batches** (default: 60 tests)
  - Each test gets **its own matrix** (no waste!)
  - **True parallel compute**: queues all operations, single sync at end
  - **Adaptive**: excludes largest test on OOM and retries batch
- **Pro**:
  - ✅ Maximum speed via batching
  - ✅ Memory-efficient (per-test matrices)
  - ✅ Adaptive OOM handling
  - ✅ Includes aggressive cleanup
- **Con**: Slightly more complex, requires more GPU memory upfront
- **Best For**: Large datasets with sufficient GPU memory, speed-critical analysis

---

## 🎯 Which Version Should I Use?

| Scenario | Recommended Version | Why |
|----------|-------------------|-----|
| **Testing/debugging** | **Version 1** | Simpler, easier to debug |
| **really small datasets**  | **cpu version** | it does the job, you can parallelize on multiple cores if you have enough cores and free ram
| **moderate size datasets that fit completely in the gpu vram** | **Version 1** | Simpler, easier to debug |
| **large size datasets  too big for the vram** | **Version 2** | Fastest option available |
---

## 🧹 GPU Memory Management

**Problem**: GPU memory can accumulate between runs due to CuPy's memory pool caching.

**Solution**: Use `gpu_cleanup()` between function runs:

```r
# Run analysis
results <- compute_Qx_batch_gpu_adaptive(...)

# Clean GPU memory
gpu_cleanup()

# Run again with fresh GPU memory
results2 <- compute_Qx_batch_gpu_adaptive(...)
```

**What it does**:
- Synchronizes GPU streams
- Releases cached memory back to GPU OS
- Runs garbage collection
- No device reset needed!

---


# GPU-Accelerated QX Test for Population Genetics

## Overview

This notebook demonstrates a **GPU-accelerated implementation** of the **QX variance test** for detecting differences in polygenic selection between populations.

### The QX Test

The QX test (Joshi et al., *Nature* 2015) measures heterogeneity in polygenic scores between populations by comparing:
- **Observed variance** in allele frequency × effect size products
- **Expected variance** under the null hypothesis (via permutation testing)

**Reference:**  
Joshi PK et al. (2015) *Directional dominance on stature and cognition in diverse human populations.*  
Nature. [doi:10.1038/nature14618](https://doi.org/10.1038/nature14618)




## ⚠️ Demo Data Notice

This notebook uses **simulated data** to demonstrate the GPU optimization technique.  
Real genomic data is subject to data use agreements and is not included.

## 1. Setup & Dependencies

### Installation (First Time Only)

If you don't have CuPy installed, run this cell first:

```r
# Install CuPy via pip (one-time setup)
reticulate::py_install("cupy-cuda12x")  # For CUDA 12.x
# OR for CUDA 11.x use: reticulate::py_install("cupy-cuda11x")
```

**Note**: This requires CUDA toolkit to be installed on your system. If you get errors, check your CUDA version with `nvidia-smi` in the terminal.

In [1]:
Sys.setenv(RETICULATE_PYTHON = "D:/Users/Pasqu/anaconda3/python.exe")

In [2]:
# Load required R packages
library(reticulate)

# Import CuPy for GPU computing (Python library accessed via reticulate)
cp <- import("cupy")

cat("✅ CuPy version:", cp$`__version__`, "\n")
cat("✅ GPU available:", cp$cuda$is_available(), "\n")

if (cp$cuda$is_available()) {
  # Get GPU device count and memory info
  device_count <- cp$cuda$runtime$getDeviceCount()
  cat("✅ GPU devices found:", device_count, "\n")
  
  # Get current device properties
  device_id <- cp$cuda$Device()$id
  cat("✅ Using GPU device:", device_id, "\n")
}

Warning message:
"il pacchetto 'reticulate' è stato creato con R versione 4.4.3"


✅ CuPy version: 13.6.0 
✅ GPU available: TRUE 
✅ GPU devices found: 1 
✅ Using GPU device: 0 


## 2. Generate Simulated Data

We'll create realistic simulated genetic data mimicking:
- **SNP effect sizes** (beta coefficients from GWAS)
- **Population allele frequencies** (MAFs) for two populations
- **Population structure** (different MAF distributions)
- **Missing data** (~2% NAs, realistic for GWAS quality control)

### NA Handling:

Real GWAS data often has missing values due to:
- Low sequencing coverage for some SNPs
- Quality control filters removing problematic genotypes
- Population-specific genotyping issues

Following the professor's implementation in `Tst_QX_Computation.r`:
1. **Pre-filter**: Remove entire SNPs (rows) that have NA in either MAF or beta using `na.omit()`
2. **Consistency**: This ensures observed and permuted calculations use identical SNP sets
3. **Safety**: Keep `na.rm = TRUE` in sums as a failsafe (shouldn't trigger after pre-filtering)

In [3]:
# Function to generate realistic simulated data
generate_simulated_data <- function(n_snps = 500, 
                                    pop1_mean_maf = 0.3, 
                                    pop2_mean_maf = 0.25,
                                    effect_sd = 0.05,
                                    na_rate = 0.02) {  # 2% missing data (realistic for GWAS)
  
  # Generate random test name and ID (simulating real GWAS traits)
  trait_names <- c("Standing_height", "Body_mass_index", "Skin_colour", "Eye_colour", 
                   "Hair_colour", "Mean_corpuscular_volume", "Red_blood_cell_count",
                   "Platelet_count", "White_blood_cell_count", "Hemoglobin_concentration",
                   "Cholesterol_total", "HDL_cholesterol", "LDL_cholesterol", "Triglycerides",
                   "Glucose", "C-reactive_protein", "Vitamin_D", "Bone_mineral_density")
  
  categories <- c("continuous", "biomarkers", "categorical", "physical_measures")
  
  # Randomly select trait and category
  test_name <- sample(trait_names, 1)
  category <- sample(categories, 1)
  trait_number <- sample(1000:9999, 1)
  sex_category <- sample(c("both_sexes", "male", "female"), 1)
  
  # Create test ID
  test_id <- paste0(category, "-", trait_number, "-", sex_category)
  
  # Generate SNP effect sizes (beta coefficients)
  # Realistic distribution: most small effects, few large effects
  betas <- rnorm(n_snps, mean = 0, sd = effect_sd)
  
  # Generate population 1 MAFs (minor allele frequencies)
  # Beta distribution gives realistic MAF distribution
  maf1 <- rbeta(n_snps, shape1 = 2, shape2 = 5)
  maf1 <- pmin(pmax(maf1, 0.01), 0.5)  # Keep in valid range
  
  # Generate population 2 MAFs with some divergence from pop1
  # Add noise to create population structure
  maf2 <- maf1 + rnorm(n_snps, mean = pop2_mean_maf - pop1_mean_maf, sd = 0.05)
  maf2 <- pmin(pmax(maf2, 0.01), 0.5)  # Keep in valid range
  
  # Introduce realistic missing data (NA values)
  # In real GWAS data, some SNPs may have missing MAF due to:
  # - Low coverage in sequencing
  # - Quality control filters
  # - Population-specific genotyping issues
  if (na_rate > 0) {
    n_missing <- ceiling(n_snps * na_rate)
    
    # Randomly assign NAs to MAF1 (some SNPs missing in pop1)
    na_indices_maf1 <- sample(1:n_snps, n_missing, replace = FALSE)
    maf1[na_indices_maf1] <- NA
    
    # Randomly assign NAs to MAF2 (some SNPs missing in pop2)
    # Use different indices to simulate realistic missing patterns
    na_indices_maf2 <- sample(1:n_snps, n_missing, replace = FALSE)
    maf2[na_indices_maf2] <- NA
  }
  
  return(list(
    maf1 = maf1,
    maf2 = maf2,
    betas = betas,
    test_name = test_name,
    test_id = test_id
  ))
}




In [4]:
# Generate multiple test datasets
n_tests <- 300  # INCREASE to see batch advantage!
n_perm <- 10000
seed <- 123

cat("Generating", n_tests, "simulated population comparisons...\n")

test_data <- vector("list", n_tests)
set.seed(seed)

for (i in 1:n_tests) {
  # Vary parameters slightly for each test
  n_snps <- sample(400:600, 1)
  pop1_maf <- runif(1, 0.25, 0.35)
  pop2_maf <- runif(1, 0.2, 0.3)
  
  test_data[[i]] <- generate_simulated_data(
    n_snps = n_snps,
    pop1_mean_maf = pop1_maf,
    pop2_mean_maf = pop2_maf
  )
}

cat("✅ Generated", n_tests, "test datasets\n")

# Prepare data lists
maf1_list <- lapply(test_data, function(x) x$maf1)
maf2_list <- lapply(test_data, function(x) x$maf2)
beta_list <- lapply(test_data, function(x) x$betas)
test_names <- sapply(test_data, function(x) x$test_name)
test_ids <- sapply(test_data, function(x) x$test_id)

# Show first few test identifiers
cat("\nFirst 5 test identifiers:\n")
for (i in 1:min(5, n_tests)) {
  cat(sprintf("  Test %d: %s (%s)\n", i, test_names[i], test_ids[i]))
}


Generating 300 simulated population comparisons...
✅ Generated 300 test datasets

First 5 test identifiers:
  Test 1: Triglycerides (categorical-3985-male)
  Test 2: LDL_cholesterol (biomarkers-7634-both_sexes)
  Test 3: Platelet_count (continuous-5453-female)
  Test 4: Cholesterol_total (physical_measures-2188-both_sexes)
  Test 5: Standing_height (continuous-6524-both_sexes)


## 3. CPU Baseline

This is the reference implementation used for validation. All GPU versions must produce identical `Qx_observed` values. Pls open an issue on github if you notice errors

In [5]:
# Function to compute Fst between two populations
Fst <- function(maf1, maf2, n1 = NULL, n2 = NULL) {
  if (!is.null(n1) && !is.null(n2)) {
    # Full Hudson (1992) estimator with sample size correction
    num <- (maf1 - maf2)^2 - maf1 * (1 - maf1) / n1 - maf2 * (1 - maf2) / n2
    denom <- maf1 * (1 - maf2) + maf2 * (1 - maf1) + 1e-12
    fst_vals <- ifelse(denom > 0, num / denom, 0)
  } else {
    # Simplified version (for synthetic data)
    fst_num <- (maf1 - maf2)^2
    fst_denom <- maf1 * (1 - maf2) + maf2 * (1 - maf1) + 1e-12
    fst_vals <- ifelse(fst_denom > 0, fst_num / fst_denom, 0)
  }
  return(fst_vals)
}


In [6]:
# CPU implementation of Qx statistic
# Used as baseline for validation
compute_Qx_cpu <- function(maf1, maf2, beta, 
                            neutral_maf1 = NULL, neutral_maf2 = NULL, n1 = NULL, n2 = NULL, 
                            n_perm = 10000, seed = NULL) {
  if (!is.null(seed)) set.seed(seed)
 
  # Remove rows with ANY NA
  data_complete <- na.omit(data.frame(maf1 = maf1, maf2 = maf2, beta = beta))
  maf1 <- data_complete$maf1
  maf2 <- data_complete$maf2
  beta <- data_complete$beta
 
  # Calculate observed Qx statistic
  diff_obs <- maf1 * beta - maf2 * beta
  N_Qx_obs <- sum(diff_obs)^2
 
  # Denominator (constant across permutations)  
  if (!is.null(neutral_maf1) && !is.null(neutral_maf2)) {
    keep_neut <- complete.cases(neutral_maf1, neutral_maf2)
    n1_neutral <- neutral_maf1[keep_neut]
    n2_neutral <- neutral_maf2[keep_neut]
    Fst_neutral <- mean(Fst(n1_neutral, n2_neutral, n1 = n1, n2 = n2), na.rm = TRUE)
    Va_term <- sum(beta^2 * 2 * maf1 * maf2)
    D_Qx <- Va_term * Fst_neutral
  } else {
    fst_values <- Fst(maf1, maf2, n1 = n1, n2 = n2)
    D_Qx <- sum((beta^2) * (2 * maf1 * maf2) * fst_values)
  }
 
  Qx_obs <- N_Qx_obs / D_Qx
 
  # Fst observed
  Fst_obs <- ifelse(!is.null(neutral_maf1) && !is.null(neutral_maf2),
                    Fst_neutral,
                    mean(fst_values, na.rm = TRUE))
 
  # Permutation loop
  N_Qx_perm <- numeric(n_perm)
  for (i in 1:n_perm) {
    beta_perm <- sample(beta)
    diff_perm <- maf1 * beta_perm - maf2 * beta_perm
    N_Qx_perm[i] <- sum(diff_perm)^2
  }
 
  # Compute statistics
  Qx_perm <- N_Qx_perm / D_Qx
 
  # p-value: proportion of permuted |Qx| >= observed |Qx|
  empirical_p <- mean(abs(Qx_perm) >= abs(Qx_obs))
 
  return(list(
    Qx = Qx_obs,
    Fst = Fst_obs,
    p_value = empirical_p
  ))
}

## Version 1: Sequential GPU 

Generates permutation matrix **directly on GPU** using CuPy's random functions:
- Each test gets a **perfectly-sized** `n_snps × n_perm` matrix
- No CPU→GPU transfer overhead for permutation matrix
- Memory-efficient: no waste from oversized matrices
- **~10x faster than CPU**


### Best for:
- small or moderate sized datasets that can fit comfortably in the gpu vram
- testing and debugging purposes since it's easier to comprehend

In [7]:
# Version 1 ADAPTIVE: Sequential GPU with PURE GPU + ADAPTIVE OOM HANDLING
# Generates permutation matrix on GPU AND handles OOM by excluding the test!

compute_Qx_sequential_gpu_pure_adaptive <- function(maf1, maf2, beta,neutral_maf1 = NULL, neutral_maf2 = NULL,n1 = NULL, n2 = NULL,  n_perm = 10000, seed = NULL,
                                                     test_name = NULL, test_id = NULL) {
  
  # Wrap entire computation in tryCatch for OOM handling
  result <- tryCatch({
    
    if (!is.null(seed)) set.seed(seed)
    
# Remove rows with ANY NA FIRST (following professor's approach)
    data_complete <- na.omit(data.frame(maf1 = maf1, maf2 = maf2, beta = beta))
    maf1 <- data_complete$maf1
    maf2 <- data_complete$maf2
    beta <- data_complete$beta
    
    n <- length(beta)
    
    if (n == 0) {
      return(list(
        Qx = NA, Fst = NA, p_value = NA,
        test_name = test_name, test_id = test_id,
        n_snps = 0, error = "No valid SNPs after NA removal",
        excluded = FALSE
      ))
    }
    
    # Transfer data to GPU
    maf1_gpu <- cp$array(maf1, dtype = cp$float64)
    maf2_gpu <- cp$array(maf2, dtype = cp$float64)
    beta_gpu <- cp$array(beta, dtype = cp$float64)
    # Add n1/n2 if provided
    n1_gpu <- if (!is.null(n1)) cp$array(rep(n1, length(maf1)), dtype = cp$float64) else NULL
    n2_gpu <- if (!is.null(n2)) cp$array(rep(n2, length(maf2)), dtype = cp$float64) else NULL

    # ===== OBSERVED Qx (matching CPU formula) =====
    diff_obs <- maf1_gpu * beta_gpu - maf2_gpu * beta_gpu
    N_Qx_obs <- cp$sum(diff_obs)**2
    
    # Denominator: sum(beta^2 * (2 * maf1 * maf2) * fst_vals)
    if (!is.null(neutral_maf1) && !is.null(neutral_maf2)) {
      # --- NEUTRAL MODE: global Fst from neutral SNPs ---
      neut_keep <- complete.cases(neutral_maf1, neutral_maf2)
      neu1 <- cp$array(neutral_maf1[neut_keep], dtype = cp$float64)
      neu2 <- cp$array(neutral_maf2[neut_keep], dtype = cp$float64)
      
      if (!is.null(n1_gpu) && !is.null(n2_gpu)) {
        fst_num <- (maf1_gpu - maf2_gpu)**2 - maf1_gpu * (1 - maf1_gpu) / n1_gpu - maf2_gpu * (1 - maf2_gpu) / n2_gpu
      } else {
        fst_num <- (maf1_gpu - maf2_gpu)**2
      }
      fst_denom <- neu1 * (1 - neu2) + neu2 * (1 - neu1) + 1e-12
      fst_neutral_global <- cp$mean(fst_num / fst_denom)
      
      Va_term <- cp$sum(beta_gpu**2 * 2 * maf1_gpu * maf2_gpu)
      D_Qx <- Va_term * fst_neutral_global
      
      # Fst observed = neutral global Fst
      Fst_obs <- as.numeric(fst_neutral_global)
      
    } else {
      # --- STANDARD MODE: per-locus Fst from GWAS SNPs ---
      if (!is.null(n1_gpu) && !is.null(n2_gpu)) {
        fst_num <- (maf1_gpu - maf2_gpu)**2 - maf1_gpu * (1 - maf1_gpu) / n1_gpu - maf2_gpu * (1 - maf2_gpu) / n2_gpu
      } else {
        fst_num <- (maf1_gpu - maf2_gpu)**2
      }
      fst_denom <- maf1_gpu * (1 - maf2_gpu) + maf2_gpu * (1 - maf1_gpu) + 1e-12
      fst_vals <- cp$where(fst_denom > 0, fst_num / fst_denom, 0)
      
      D_Qx <- cp$sum((beta_gpu**2) * (2 * maf1_gpu * maf2_gpu) * fst_vals)
      
      # Fst observed = mean of per-locus Hudson Fst (coerente)
      Fst_obs <- cp$mean(fst_vals)
    }
    
    # Qx observed
    Qx_obs <- N_Qx_obs / D_Qx
    #Qx_obs <- as.numeric(Qx_obs)

    # ===== PERMUTATION TEST - PURE GPU! =====
    # Generate permutation indices DIRECTLY ON GPU!
    if (!is.null(seed)) {
      cp$random$seed(as.integer(seed))
    }
    
    n_gpu <- as.integer(n)
    n_perm_gpu <- as.integer(n_perm)
    
    # Generate random matrix on GPU and argsort to get permutation indices
    random_matrix <- cp$random$rand(n_gpu, n_perm_gpu, dtype = cp$float32)
    perm_indices_all <- cp$argsort(random_matrix, axis = 0L)
    rm(random_matrix)
    cp$cuda$Stream$null$synchronize()
    cp$get_default_memory_pool()$free_all_blocks()
    
    # Permute beta using the GPU indices (stays on GPU!)
    beta_perm_all <- beta_gpu[perm_indices_all]  # Shape: (n, n_perm)
    
    # Free permutation indices
    rm(perm_indices_all)
    cp$cuda$Stream$null$synchronize()
    cp$get_default_memory_pool()$free_all_blocks()
    
    # ===== COMPUTE PERMUTED Qx (all on GPU) =====
    # Expand MAFs for broadcasting
    maf1_expanded <- cp$expand_dims(maf1_gpu, axis = 1L)
    maf2_expanded <- cp$expand_dims(maf2_gpu, axis = 1L)
    
    maf1_beta_perm <- maf1_expanded * beta_perm_all
    maf2_beta_perm <- maf2_expanded * beta_perm_all
    
    rm(beta_perm_all)
    cp$cuda$Stream$null$synchronize()
    cp$get_default_memory_pool()$free_all_blocks()
    
    diff_perm_all <- maf1_beta_perm - maf2_beta_perm  # Shape: (n, n_perm)
    
    rm(maf1_beta_perm, maf2_beta_perm)
    cp$cuda$Stream$null$synchronize()
    cp$get_default_memory_pool()$free_all_blocks()
    
    # Sum over SNPs for each permutation
    diff_sum_perm <- cp$sum(diff_perm_all, axis = 0L)  # Shape: (n_perm,)
    
    rm(diff_perm_all)
    cp$cuda$Stream$null$synchronize()
    cp$get_default_memory_pool()$free_all_blocks()
    
    # Square the sum to get N_Qx_perm
    N_Qx_perm <- diff_sum_perm**2  # Shape: (n_perm,)
    
    # Compute Qx_perm = N_Qx_perm / D_Qx (same denominator!)
    Qx_perm_all <- N_Qx_perm / D_Qx
    
    # ===== P-VALUE (on GPU) =====
    Qx_perm_abs <- cp$abs(Qx_perm_all)
    Qx_obs_abs <- cp$abs(Qx_obs)
    
    exceeds <- Qx_perm_abs >= Qx_obs_abs
    p_value_gpu <- cp$mean(exceeds$astype(cp$float64))
    
    # Transfer final results to CPU
    Qx_final <- as.numeric(Qx_obs$get())
    Fst_final <- as.numeric(Fst_obs$get())
    p_value_final <- as.numeric(p_value_gpu$get())
    
    # Cleanup GPU memory
    rm(maf1_gpu, maf2_gpu, beta_gpu, Qx_perm_all,fst_num,fst_denom,fst_vals,D_Qx,N_Qx_perm,Qx_obs_abs,diff_sum_perm,exceeds)
    if (exists("fst_neutral_global")) {
      rm(fst_neutral_global,fst_denom_n,fst_num_n,Va_term)
    }
    cp$cuda$Stream$null$synchronize()
    cp$get_default_memory_pool()$free_all_blocks()
    
    # Success!
    list(
      Qx = Qx_final,
      Fst = Fst_final,
      p_value = p_value_final,
      test_name = test_name,
      test_id = test_id,
      n_snps = n,
      excluded = FALSE
    )
    
  }, error = function(e) {
    
    # Check for memory errors (OOM)
    error_class <- class(e)
    error_msg <- conditionMessage(e)
    
    is_python_memory <- any(grepl("MemoryError", error_class, ignore.case = TRUE))
    is_cuda_oom <- any(grepl("CUDARuntimeError|cudaErrorMemoryAllocation", error_class, ignore.case = TRUE))
    is_r_error <- any(error_class == "error")
    
    is_oom <- is_python_memory || is_cuda_oom || is_r_error
    
    if (is_oom) {
      # OOM ERROR - EXCLUDE THIS TEST!
      cat(sprintf("  ⚠️  OUT OF MEMORY for test: %s (%s) with %d SNPs\n", 
                  test_name, test_id, length(beta)))
      cat(sprintf("      Error: %s\n", substr(error_msg, 1, 150)))
      cat("      Test EXCLUDED!\n\n")
      
      # Aggressive GPU cleanup
      tryCatch({
        cp$cuda$Stream$null$synchronize()
        mempool <- cp$get_default_memory_pool()
        pinned_mempool <- cp$get_default_pinned_memory_pool()
        mempool$free_all_blocks()
        pinned_mempool$free_all_blocks()
      }, error = function(cleanup_error) {
        # GPU cleanup failed, continue anyway
      })
      
      gc()
      gc()
      
      # Return excluded status
      return(list(
        Qx = NA,
        Fst = NA,
        p_value = NA,
        test_name = test_name,
        test_id = test_id,
        n_snps = length(beta),
        excluded = TRUE,
        error = "OUT_OF_MEMORY"
      ))
      
    } else {
      # Not an OOM error - re-throw
      cat(sprintf("  ❌ ERROR for test: %s (%s) (not OOM-related)\n", 
                  test_name, test_id))
      cat(sprintf("      %s\n\n", error_msg))
      stop(e)
    }
  })
  
  return(result)
}

cat("✅ Version 10 ADAPTIVE: Sequential GPU PURE with OOM handling defined\n")
cat("   - Generates permutation matrix on GPU (no CPU transfers)\n")
cat("   - Adaptive: catches OOM errors and excludes the test\n")
cat("   - Returns excluded=TRUE flag for tests that don't fit in GPU memory\n")
cat("   - Aggressive GPU cleanup after OOM\n")


✅ Version 10 ADAPTIVE: Sequential GPU PURE with OOM handling defined


   - Generates permutation matrix on GPU (no CPU transfers)
   - Adaptive: catches OOM errors and excludes the test
   - Returns excluded=TRUE flag for tests that don't fit in GPU memory
   - Aggressive GPU cleanup after OOM


## Version 11: Batch GPU with Per-Test Matrices 🚀

**The FASTEST implementation!** Combines batch processing with memory efficiency.

### Key Innovation:
**Version 11 generates a perfectly-sized matrix for EACH test** and processes them in parallel.

### How it works:
1. **For each test in batch**: Generate exact `n_snps × n_perm` matrix on GPU
2. **Queue all GPU operations** without synchronization (true parallelism!)
3. **Single sync point** at end to transfer all results at once
4. **Adaptive OOM handling**: Excludes largest test from batch and retries
5. **Aggressive cleanup**: Proper GPU memory management between batches

### Performance:
- **Memory efficient**: No waste from oversized matrices
- **Scalable**: Handles heterogeneous datasets with varying SNP counts

### Best for:
- Large datasets (100+ tests)
- Speed-critical analysis

In [8]:
# Version 11: Batch GPU with PER-TEST matrices (no waste!)
# Each test gets its own perfectly-sized permutation matrix
# TRUE parallel compute: queue all ops, single sync at end

compute_Qx_batch_gpu_adaptive <- function(maf1, maf2, beta,neutral_maf1 = NULL, neutral_maf2 = NULL,  
                                                   test_names = NULL, test_ids = NULL,n1 = NULL, n2 = NULL,
                                                   n_perm = 10000, seed = NULL,
                                                   batch_size = 60,
                                                   max_removal_attempts = 50) {
  
  n_tests_original <- length(maf1)
  
  cat(strrep("=", 80), "\n")
  cat("VERSION 11: BATCH GPU WITH PER-TEST MATRICES\n")
  cat(strrep("=", 80), "\n\n")
  cat(sprintf("Starting with %d tests, batch size: %d, permutations: %d\n", 
              n_tests_original, batch_size, n_perm))
  
  # Initialize outputs
  all_results <- vector("list", n_tests_original)
  excluded_tests <- list()
  
  # Default test identifiers if not provided
  if (is.null(test_names)) test_names <- paste0("Test_", 1:n_tests_original)
  if (is.null(test_ids)) test_ids <- paste0("ID_", 1:n_tests_original)
  
  # Pre-filter NAs from all tests
  cat("Pre-filtering NAs from all tests...\n")
  for (i in 1:n_tests_original) {
    data_complete <- na.omit(data.frame(
      maf1 = maf1[[i]], 
      maf2 = maf2[[i]], 
      beta = beta[[i]]
    ))
    maf1[[i]] <- data_complete$maf1
    maf2[[i]] <- data_complete$maf2
    beta[[i]] <- data_complete$beta
  }

  # =================================================================
  # PRE-FILTERING NAs: NEUTRAL data (solo se passati)
  # =================================================================
  use_neutral <- !is.null(neutral_maf1) && !is.null(neutral_maf2)
  
  if (use_neutral) {
    if (length(neutral_maf1) != n_tests_original || length(neutral_maf2) != n_tests_original) {
      stop("neutral_maf1 and neutral_maf2 must have the same length as maf1/maf2/beta")
    }
    
    cat("Pre-filtering NAs from neutral SNP data...\n")
    for (i in 1:n_tests_original) {
      neut_complete <- na.omit(data.frame(
        neutral_maf1 = neutral_maf1[[i]],
        neutral_maf2 = neutral_maf2[[i]]
      ))
      neutral_maf1[[i]] <- neut_complete$neutral_maf1
      neutral_maf2[[i]] <- neut_complete$neutral_maf2
    }
  } else {
    cat("No neutral SNPs provided — using per-locus F_ST from GWAS SNPs\n")
  }
  
  # Calculate SNP counts
  n_snps <- sapply(beta, length)
  
  # Process in batches
  batch_starts <- seq(1, n_tests_original, by = batch_size)
  
  for (batch_idx in seq_along(batch_starts)) {
    batch_start <- batch_starts[batch_idx]
    batch_end <- min(batch_start + batch_size - 1, n_tests_original)
    
    cat(sprintf("\n--- Batch %d/%d: Tests %d-%d ---\n", 
                batch_idx, length(batch_starts), batch_start, batch_end))
    
    # Get indices for this batch
    batch_indices <- batch_start:batch_end
    batch_removal_count <- 0
    
    # Retry loop for OOM handling
    while (length(batch_indices) > 0 && batch_removal_count < max_removal_attempts) {
      
      batch_result <- tryCatch({
        
        # Extract batch data
        maf1_batch <- maf1[batch_indices]
        maf2_batch <- maf2[batch_indices]
        beta_batch <- beta[batch_indices]
        n_snps_batch <- n_snps[batch_indices]
        n1_batch <- if (!is.null(n1)) n1[batch_indices] else NULL
        n2_batch <- if (!is.null(n2)) n2[batch_indices] else NULL

        # === SE I NEUTRALI SONO STATI PASSATI: estrai solo quelli del batch ===
        neutral1_batch <- if (use_neutral) neutral_maf1[batch_indices] else NULL
        neutral2_batch <- if (use_neutral) neutral_maf2[batch_indices] else NULL

        batch_length <- length(batch_indices)
        
        cat(sprintf("  Processing %d tests (max SNPs: %d)...\n", 
                    batch_length, max(n_snps_batch)))
        
        # ===== STAGE 1: Generate PER-TEST matrices on GPU =====
        if (!is.null(seed)) cp$random$seed(as.integer(seed + batch_idx))
        
        perm_matrices_gpu <- vector("list", batch_length)
        
        for (i in 1:batch_length) {
          n_i <- n_snps_batch[i]
          # Generate perfectly-sized matrix for THIS test
          random_matrix <- cp$random$rand(as.integer(n_i), as.integer(n_perm), dtype = cp$float32)
          perm_matrices_gpu[[i]] <- cp$argsort(random_matrix, axis = 0L)
          rm(random_matrix)
        }
        
        # ===== STAGE 2: Transfer test data to GPU =====
        maf1_gpu <- lapply(maf1_batch, function(x) cp$array(x, dtype = cp$float64))
        maf2_gpu <- lapply(maf2_batch, function(x) cp$array(x, dtype = cp$float64))
        beta_gpu <- lapply(beta_batch, function(x) cp$array(x, dtype = cp$float64))
        n1_batch_gpu <- if (!is.null(n1_batch)) lapply(n1_batch, function(x) cp$array(x, dtype = cp$int32))
        n2_batch_gpu <- if (!is.null(n2_batch)) lapply(n2_batch, function(x) cp$array(x, dtype = cp$int32))
        
        # === NEUTRALI: trasferisci sulla GPU solo se use_neutral è TRUE ===
        #neutral1_gpu <- NULL
        #neutral2_gpu <- NULL
        
        if (use_neutral) {
          cat(sprintf("  Transferring neutral SNPs for %d tests to GPU...\n", batch_length))
          neutral1_gpu <- lapply(neutral1_batch, function(x) cp$array(x, dtype = cp$float64))
          neutral2_gpu <- lapply(neutral2_batch, function(x) cp$array(x, dtype = cp$float64))
        }
        
# ===== STAGE 3: Compute ALL tests in parallel (no sync until end!) =====
        Qx_obs_list   <- vector("list", batch_length)
        Fst_obs_list  <- vector("list", batch_length)
        p_value_list  <- vector("list", batch_length)
        
        # === CALCOLO F_ST NEUTRALE (solo se use_neutral) ===
        neutral_Fst_batch <- NULL
        neutral_Fst_batch <- numeric(batch_length)

        
        # ===  calcola Qx per ogni test del batch ===
        for (i in 1:batch_length) {
          # --- Numerator observed ---
          diff_obs <- maf1_gpu[[i]] * beta_gpu[[i]] - maf2_gpu[[i]] * beta_gpu[[i]]
          N_Qx_obs <- cp$sum(diff_obs)**2
          
          # --- DENOMINATOR: neutral or per-locus ---
          if (use_neutral) {
            # Use neutral F_ST
            if (n1_batch_gpu != NULL && n2_batch_gpu != NULL) {
              fst_num_n <- ((neutral1_gpu[[i]] - neutral2_gpu[[i]])**2 - 
                            neutral1_gpu[[i]] * (1 - neutral1_gpu[[i]]) / n1_batch_gpu[[i]] - 
                            neutral2_gpu[[i]] * (1 - neutral2_gpu[[i]]) / n2_batch_gpu[[i]])
            } 
            else {
            fst_num_n <- (neutral1_gpu[[i]] - neutral2_gpu[[i]])**2 
              }
            fst_denom_n <- neutral1_gpu[[i]] * (1 - neutral2_gpu[[i]]) + neutral2_gpu[[i]] * (1 - neutral1_gpu[[i]]) + 1e-12
            neutral_Fst_batch[i] <- cp$as.numeric(cp$mean(fst_num_n / fst_denom_n))
            Va_term <- cp$sum(beta_gpu[[i]]**2 * 2 * maf1_gpu[[i]] * maf2_gpu[[i]])
            D_Qx <- Va_term * neutral_Fst_batch[i]
            Fst_obs_list[[i]] <- neutral_Fst_batch[i]
          } else {
            # Versione classica per locus (Hudson/Bhatia esatto)
            if (!is.null(n1_batch_gpu) && !is.null(n2_batch_gpu)) {
              fst_num <- (maf1_gpu[[i]] - maf2_gpu[[i]])**2 - 
                          maf1_gpu[[i]] * (1 - maf1_gpu[[i]]) / n1_batch_gpu[[i]] - 
                          maf2_gpu[[i]] * (1 - maf2_gpu[[i]]) / n2_batch_gpu[[i]]
            } else {
              fst_num <- (maf1_gpu[[i]] - maf2_gpu[[i]])**2 
            }
            #fst_num <- (maf1_gpu[[i]] - maf2_gpu[[i]])**2 
            fst_denom <- maf1_gpu[[i]] * (1 - maf2_gpu[[i]]) + maf2_gpu[[i]] * (1 - maf1_gpu[[i]]) + 1e-12
            fst_vals <- cp$where(fst_denom > 0, fst_num / fst_denom, 0)
            
            D_Qx <- cp$sum(beta_gpu[[i]]**2 * 2 * maf1_gpu[[i]] * maf2_gpu[[i]] * fst_vals)
            Fst_obs_list[[i]] <- cp$mean(fst_vals)
          }
          
          # --- Qx observed ---
          Qx_obs_list[[i]] <- N_Qx_obs / D_Qx
          
          # --- Permutations ---
          beta_perm <- beta_gpu[[i]][perm_matrices_gpu[[i]]]
          
          maf1_exp <- cp$expand_dims(maf1_gpu[[i]], axis = 1L)
          maf2_exp <- cp$expand_dims(maf2_gpu[[i]], axis = 1L)
          
          diff_perm <- maf1_exp * beta_perm - maf2_exp * beta_perm
          diff_sum_perm <- cp$sum(diff_perm, axis = 0L)
          N_Qx_perm <- diff_sum_perm**2
          Qx_perm <- N_Qx_perm / D_Qx
          
          # p-value
          p_value_list[[i]] <- cp$mean((cp$abs(Qx_perm) >= cp$abs(Qx_obs_list[[i]]))$astype(cp$float64))
        }
        # ===== SINGLE SYNC POINT: Transfer all results at once =====
        cat("  Transferring results from GPU...\n")
        
        batch_results <- lapply(1:batch_length, function(i) {
          list(
            Qx      = as.numeric(Qx_obs_list[[i]]$get()),
            Fst     = as.numeric(Fst_obs_list[[i]]$get()),      # ← neutrale o per-locus
            p_value = as.numeric(p_value_list[[i]]$get()),
            test_name = test_names[batch_indices[i]],
            test_id   = test_ids[batch_indices[i]],
            n_snps    = n_snps_batch[i],
            neutral_used = use_neutral                          # ← info utile per chi usa il tool
          )
        })
        
        # Cleanup GPU memory aggressively
        rm(perm_matrices_gpu, maf1_gpu, maf2_gpu, beta_gpu,beta_perm,diff_perm,diff_sum_perm,N_Qx_perm,Qx_perm)
        rm(Qx_obs_list, Fst_obs_list, p_value_list)
        
        cat(sprintf("  Batch completed successfully! (%d tests)\n", batch_length))
        

        # Return results
        list(success = TRUE, results = batch_results)
      }, error = function(e) {
        # OOM error handling
        cat(sprintf("  OOM Error: %s\n", e$message))
        list(success = FALSE, error = e$message)
      }, finally = {
          # CLEANUP GPU MEMORY
          if (use_neutral) {
            rm(neutral1_gpu, neutral2_gpu, neutral_Fst_batch)
          }
          cp$cuda$Stream$null$synchronize()
          mempool <- cp$get_default_memory_pool()
          mempool$free_all_free()
          pinned_mempool <- cp$get_default_pinned_memory_pool()
          pinned_mempool$free_all_blocks()
          
          if (use_neutral && exists("neutral1_gpu")) {
            rm(neutral1_gpu, neutral2_gpu, neutral_Fst_batch)
          }
          gc(); gc()
        })
      
      # Check if batch succeeded
      if (batch_result$success) {
        # Store results
        for (i in seq_along(batch_indices)) {
          all_results[[batch_indices[i]]] <- batch_result$results[[i]]
        }
        break  # Exit retry loop
        
      } else {
        # Remove largest test and retry
        batch_removal_count <- batch_removal_count + 1
        
        if (length(batch_indices) == 1) {
          cat("  Only one test left and it still fails - excluding it\n")
          excluded_tests[[length(excluded_tests) + 1]] <- list(
            index = batch_indices[1],
            test_name = test_names[batch_indices[1]],
            test_id = test_ids[batch_indices[1]],
            n_snps = n_snps[batch_indices[1]],
            error = batch_result$error
          )
          break
        }
        
        # Find and remove largest test
        batch_n_snps <- n_snps[batch_indices]
        largest_idx_in_batch <- which.max(batch_n_snps)
        largest_global_idx <- batch_indices[largest_idx_in_batch]
        
        cat(sprintf("  Removing largest test: %s (%d SNPs)\n", 
                    test_names[largest_global_idx], n_snps[largest_global_idx]))
        
        excluded_tests[[length(excluded_tests) + 1]] <- list(
          index = largest_global_idx,
          test_name = test_names[largest_global_idx],
          test_id = test_ids[largest_global_idx],
          n_snps = n_snps[largest_global_idx],
          error = batch_result$error
        )
        
        batch_indices <- batch_indices[-largest_idx_in_batch]
        cat(sprintf("  Retrying with %d tests...\n", length(batch_indices)))
      }
    }
  }
  
  # Summary
  n_successful <- sum(!sapply(all_results, is.null))
  n_excluded   <- length(excluded_tests)
  
  cat("\n")
  cat(strrep("=", 80), "\n")
  cat("FASTQX v1.0 BATCH GPU — FINAL SUMMARY\n")
  cat(strrep("=", 80), "\n")
  cat(sprintf("Successfully processed : %d / %d tests\n", n_successful, n_tests_original))
  cat(sprintf("Excluded (OOM/memory) : %d tests\n", n_excluded))
  cat(sprintf("Neutral SNPs used     : %s\n", ifelse(use_neutral, "YES", "NO")))
  
  if (n_excluded > 0) {
    cat("\nExcluded tests (removed due to GPU memory limits):\n")
    for (i in seq_len(n_excluded)) {
      exc <- excluded_tests[[i]]
      cat(sprintf("  %d. %s (%s) — %d SNPs\n", 
                  i, exc$test_name, exc$test_id, exc$n_snps))
    }
  } else {
    cat("\nAll tests completed successfully!\n")
  }
  
  cat(strrep("=", 80), "\n")
  
  return(list(
    results       = all_results,
    excluded      = excluded_tests,
    n_successful  = n_successful,
    n_excluded    = n_excluded,
    neutral_used  = use_neutral
  ))
}

## 🔧 GPU Memory Management Utilities

**The GPU Memory Problem:**
CuPy's memory pool caches memory for performance, which causes residual memory to remain allocated even after `free_all_free()`. This is by design but can accumulate over multiple runs.

**Solutions:**

In [9]:
# GPU CLEANUP - Run this cell BETWEEN function runs to free ALL GPU memory
# This successfully frees cached GPU memory!

gpu_cleanup <- function() {
  cat("🔥 GPU MEMORY CLEANUP\n\n")
  
  tryCatch({
    cat("Step 1: Synchronizing GPU...\n")
    cp$cuda$Stream$null$synchronize()
    
    cat("Step 2: Freeing memory pools...\n")
    # Regular memory pool: release free chunks back to GPU OS
    mempool <- cp$get_default_memory_pool()
    mempool$free_all_free()
    
    # Pinned memory pool: free all blocks
    pinned_mempool <- cp$get_default_pinned_memory_pool()
    pinned_mempool$free_all_blocks()
    
    cat("Step 3: CPU garbage collection...\n")
    gc(); gc(); gc()
    
    cat("\n✅ GPU memory cleaned successfully!\n")
    cat("ℹ️  Memory should be fully released (check nvidia-smi)\n")
    
  }, error = function(e) {
    cat("\n❌ Cleanup failed:\n")
    cat(sprintf("   %s\n", conditionMessage(e)))
    cat("\n💡 Try restarting R session if problems persist\n")
  })
}

cat("✅ GPU cleanup function defined\n")
cat("   Usage: Run gpu_cleanup() between function runs\n")
cat("   This will free cached GPU memory without resetting the device!\n")


✅ GPU cleanup function defined
   Usage: Run gpu_cleanup() between function runs
   This will free cached GPU memory without resetting the device!


## 6. Run All Three Versions and Verify Results Match

Let's run CPU baseline, Sequential GPU (V10), and Batch GPU (V11) on the original 300-test dataset and verify they produce identical Qx values.

In [10]:
# Run CPU baseline on 300 tests (for validation)
num=300
cat(sprintf("Running CPU baseline on first %d tests...\n", num))
t_cpu <- system.time({
  results_cpu <- lapply(1:num, function(i) {
    cat(sprintf("  CPU test %d/%d...\n", i, num))
    compute_Qx_cpu(
      maf1_list[[i]], 
      maf2_list[[i]], 
      beta_list[[i]], 
      n_perm = n_perm, 
      seed = seed + i
    )
  })
})

cat(sprintf("\n✅ CPU completed in %.2f seconds\n\n", t_cpu[3]))

Running CPU baseline on first 300 tests...
  CPU test 1/300...
  CPU test 2/300...
  CPU test 3/300...
  CPU test 4/300...
  CPU test 5/300...
  CPU test 6/300...
  CPU test 7/300...
  CPU test 8/300...
  CPU test 9/300...
  CPU test 10/300...
  CPU test 11/300...
  CPU test 12/300...
  CPU test 13/300...
  CPU test 14/300...
  CPU test 15/300...
  CPU test 16/300...
  CPU test 17/300...
  CPU test 18/300...
  CPU test 19/300...
  CPU test 20/300...
  CPU test 21/300...
  CPU test 22/300...
  CPU test 23/300...
  CPU test 24/300...
  CPU test 25/300...
  CPU test 26/300...
  CPU test 27/300...
  CPU test 28/300...
  CPU test 29/300...
  CPU test 30/300...
  CPU test 31/300...
  CPU test 32/300...
  CPU test 33/300...
  CPU test 34/300...
  CPU test 35/300...
  CPU test 36/300...
  CPU test 37/300...
  CPU test 38/300...
  CPU test 39/300...
  CPU test 40/300...
  CPU test 41/300...
  CPU test 42/300...
  CPU test 43/300...
  CPU test 44/300...
  CPU test 45/300...
  CPU test 46/300...


In [11]:
# Run Sequential GPU (V10) on ALL 300 tests
cat("Running Sequential GPU (V10) on all 300 tests...\n")
gpu_cleanup()  # Clean GPU memory first

t_seq <- system.time({
  results_seq <- lapply(1:n_tests, function(i) {
    if (i %% 50 == 0) cat(sprintf("  Sequential GPU test %d/%d...\n", i, n_tests))
    compute_Qx_sequential_gpu_pure_adaptive(
      maf1_list[[i]], 
      maf2_list[[i]], 
      beta_list[[i]], 
      n_perm = n_perm, 
      seed = seed + i,
      test_name = test_names[i],
      test_id = test_ids[i]
    )
  })
})


Running Sequential GPU (V10) on all 300 tests...
🔥 GPU MEMORY CLEANUP

Step 1: Synchronizing GPU...
Step 2: Freeing memory pools...
Step 3: CPU garbage collection...

✅ GPU memory cleaned successfully!
ℹ️  Memory should be fully released (check nvidia-smi)
  Sequential GPU test 50/300...
  Sequential GPU test 100/300...
  Sequential GPU test 150/300...
  Sequential GPU test 200/300...
  Sequential GPU test 250/300...
  Sequential GPU test 300/300...


In [12]:

cat(sprintf("\n✅ Sequential GPU completed in %.2f seconds (%.1fx speedup vs CPU )\n\n", 
            t_seq[3], (t_cpu[3] / num * n_tests) / t_seq[3]))


✅ Sequential GPU completed in 10.27 seconds (10.1x speedup vs CPU )



In [13]:
# Run Batch GPU (V11) on ALL 300 tests
cat("Running Batch GPU (V11) on all 300 tests...\n")
gpu_cleanup()  # Clean GPU memory first

t_batch <- system.time({
  results_batch_output <- compute_Qx_batch_gpu_adaptive(
    maf1_list, 
    maf2_list, 
    beta_list,
    test_names = test_names,
    test_ids = test_ids,
    n_perm = n_perm, 
    seed = seed,
    batch_size = 90
  )
})

# Extract results from batch output
results_batch <- results_batch_output$results


Running Batch GPU (V11) on all 300 tests...


🔥 GPU MEMORY CLEANUP

Step 1: Synchronizing GPU...
Step 2: Freeing memory pools...
Step 3: CPU garbage collection...

✅ GPU memory cleaned successfully!
ℹ️  Memory should be fully released (check nvidia-smi)
VERSION 11: BATCH GPU WITH PER-TEST MATRICES

Starting with 300 tests, batch size: 90, permutations: 10000
Pre-filtering NAs from all tests...
No neutral SNPs provided — using per-locus F_ST from GWAS SNPs

--- Batch 1/4: Tests 1-90 ---
  Processing 90 tests (max SNPs: 574)...
  Transferring results from GPU...
  Batch completed successfully! (90 tests)

--- Batch 2/4: Tests 91-180 ---
  Processing 90 tests (max SNPs: 571)...
  Transferring results from GPU...
  Batch completed successfully! (90 tests)

--- Batch 3/4: Tests 181-270 ---
  Processing 90 tests (max SNPs: 572)...
  Transferring results from GPU...
  Batch completed successfully! (90 tests)

--- Batch 4/4: Tests 271-300 ---
  Processing 30 tests (max SNPs: 564)...
  Transferring results from GPU...
  Batch completed suc

In [14]:

cat(sprintf("\n✅ Batch GPU completed in %.2f seconds (%.1fx speedup vs CPU , %.1fx speedup vs Sequential)\n\n", 
            t_batch[3], 
            (t_cpu[3] / num * n_tests) / t_batch[3],
            t_seq[3] / t_batch[3]))


✅ Batch GPU completed in 7.24 seconds (14.3x speedup vs CPU , 1.4x speedup vs Sequential)



In [15]:
cat(strrep("=", 80), "\n")
cat("ACCURACY VERIFICATION: CPU vs Sequential GPU vs Batch GPU\n")
cat(strrep("=", 80), "\n\n")

cat("⚠️ NOTE: Different methods use different random seeds for permutations,\n")
cat("so p-values will differ due to sampling variation.\n")
cat("However, Qx (observed statistic) MUST match perfectly across all methods!\n\n")
# Verify first 5 tests (where we have CPU results)
cat("Comparing first %d tests across all three methods:\n\n",num)

all_match <- TRUE
max_diff_cpu_seq <- 0
max_diff_cpu_batch <- 0
max_diff_seq_batch <- 0

for (i in 1:num) {
  cpu_result <- results_cpu[[i]]
  seq_result <- results_seq[[i]]
  batch_result <- results_batch[[i]]
  
  # Extract Qx values
  qx_cpu <- cpu_result$Qx
  qx_seq <- seq_result$Qx
  qx_batch <- batch_result$Qx
  
  # Calculate differences
  diff_cpu_seq <- abs(qx_cpu - qx_seq)
  diff_cpu_batch <- abs(qx_cpu - qx_batch)
  diff_seq_batch <- abs(qx_seq - qx_batch)
  
  # Track max differences
  max_diff_cpu_seq <- max(max_diff_cpu_seq, diff_cpu_seq, na.rm = TRUE)
  max_diff_cpu_batch <- max(max_diff_cpu_batch, diff_cpu_batch, na.rm = TRUE)
  max_diff_seq_batch <- max(max_diff_seq_batch, diff_seq_batch, na.rm = TRUE)
  
  # Check if all match (tolerance: 1e-5 for floating point precision)
  qx_match <- (diff_cpu_seq < 1e-5) && (diff_cpu_batch < 1e-5) && (diff_seq_batch < 1e-5)
  all_match <- all_match && qx_match
  
  cat(sprintf("Test %d: %s\n", i, test_names[i]))
  cat(sprintf("  Qx observed:\n"))
  cat(sprintf("    CPU:            %.10f\n", qx_cpu))
  cat(sprintf("    Sequential GPU: %.10f (diff: %.2e)\n", qx_seq, diff_cpu_seq))
  cat(sprintf("    Batch GPU:      %.10f (diff: %.2e)\n", qx_batch, diff_cpu_batch))
  cat(sprintf("    Match: %s\n", ifelse(qx_match, "✅", "❌")))
  
  # Show p-values for reference (expected to differ due to different permutations)
  cat(sprintf("  p-values (expected to differ - different random seeds):\n"))
  cat(sprintf("    CPU:            %.6f\n", cpu_result$p_value))
  cat(sprintf("    Sequential GPU: %.6f\n", seq_result$p_value))
  cat(sprintf("    Batch GPU:      %.6f\n\n", batch_result$p_value))
}

cat(strrep("-", 80), "\n")
cat("SUMMARY:\n")
cat(sprintf("  Max Qx difference (CPU vs Sequential GPU):  %.2e\n", max_diff_cpu_seq))
cat(sprintf("  Max Qx difference (CPU vs Batch GPU):       %.2e\n", max_diff_cpu_batch))
cat(sprintf("  Max Qx difference (Sequential vs Batch GPU): %.2e\n\n", max_diff_seq_batch))

if (all_match) {
  cat("✅ VERIFICATION PASSED!\n")
  cat("   All Qx values match across CPU, Sequential GPU, and Batch GPU (< 1e-5 tolerance)\n")
  cat("   Tiny differences are due to CPU vs GPU floating-point rounding only.\n\n")
} else {
  cat("❌ VERIFICATION FAILED!\n")
  cat("   Qx values differ significantly - check implementation!\n\n")
}

# Additional verification: Check Sequential vs Batch for ALL 300 tests
cat(strrep("-", 80), "\n")
cat("EXTENDED VERIFICATION: Sequential GPU vs Batch GPU (all 300 tests)\n")
cat(strrep("-", 80), "\n\n")

all_seq_batch_match <- TRUE
max_diff_all <- 0

for (i in 1:n_tests) {
  qx_seq <- results_seq[[i]]$Qx
  qx_batch <- results_batch[[i]]$Qx
  
  diff <- abs(qx_seq - qx_batch)
  max_diff_all <- max(max_diff_all, diff, na.rm = TRUE)
  
  if (diff >= 1e-5) {
    all_seq_batch_match <- FALSE
    cat(sprintf("  Test %d: Qx differs by %.2e\n", i, diff))
  }
}

cat(sprintf("\nMax Qx difference across all 300 tests: %.2e\n", max_diff_all))

if (all_seq_batch_match) {
  cat("✅ ALL 300 TESTS MATCH between Sequential and Batch GPU!\n")
  cat("   Sequential and Batch implementations are numerically identical.\n")
} else {
  cat("❌ Some tests differ between Sequential and Batch GPU\n")
}

ACCURACY VERIFICATION: CPU vs Sequential GPU vs Batch GPU

⚠️ NOTE: Different methods use different random seeds for permutations,
so p-values will differ due to sampling variation.
However, Qx (observed statistic) MUST match perfectly across all methods!

Comparing first %d tests across all three methods:

 300Test 1: Triglycerides
  Qx observed:
    CPU:            0.3198085366
    Sequential GPU: 0.3198085366 (diff: 2.78e-16)
    Batch GPU:      0.3198085366 (diff: 2.22e-16)
    Match: ✅
  p-values (expected to differ - different random seeds):
    CPU:            0.925200
    Sequential GPU: 0.923400
    Batch GPU:      0.923400

Test 2: LDL_cholesterol
  Qx observed:
    CPU:            11.5466349670
    Sequential GPU: 11.5466349670 (diff: 3.55e-15)
    Batch GPU:      11.5466349670 (diff: 3.55e-15)
    Match: ✅
  p-values (expected to differ - different random seeds):
    CPU:            0.041400
    Sequential GPU: 0.043600
    Batch GPU:      0.042000

Test 3: Platelet_count
 

## 7. Test Adaptive OOM Handling with Extended Dataset

Now let's test V10 Adaptive and V11 with the **extended dataset** (303 tests including 3 huge tests at strategic positions, 2 of whom causes out of memory error on my  rtx 5060 ti 16gb). This will demonstrate adaptive OOM handling (removes largest test and retries)

In [16]:
# Demonsatrate adaptive sequential GPU with the existing 300 test dataset
# PLUS add 3 HUGE tests strategically placed to trigger OOM handling in different batches

cat("Adding 3 extremely large tests at START, MIDDLE, and END...\n")

# Generate the 3 huge tests first
huge_test_start <- generate_simulated_data(n_snps = 85000, na_rate = 0.02)
cat(sprintf("  Huge test START: %s (%s) - %d SNPs\n", 
            huge_test_start$test_name, huge_test_start$test_id, length(huge_test_start$betas)))

huge_test_middle <- generate_simulated_data(n_snps = 55000, na_rate = 0.02)
cat(sprintf("  Huge test MIDDLE: %s (%s) - %d SNPs\n", 
            huge_test_middle$test_name, huge_test_middle$test_id, length(huge_test_middle$betas)))

huge_test_end <- generate_simulated_data(n_snps = 35000, na_rate = 0.02)
cat(sprintf("  Huge test END: %s (%s) - %d SNPs\n\n", 
            huge_test_end$test_name, huge_test_end$test_id, length(huge_test_end$betas)))

# Create extended lists with strategic placement
# Position 1: huge test at START (position 1)
# Position 151: huge test at MIDDLE (middle of 300 tests)
# Position 301: huge test at END (after all original tests)

maf1_list_extended <- list()
maf2_list_extended <- list()
beta_list_extended <- list()
test_names_extended <- c()
test_ids_extended <- c()

# Insert huge test at START (position 1)
maf1_list_extended[[1]] <- huge_test_start$maf1
maf2_list_extended[[1]] <- huge_test_start$maf2
beta_list_extended[[1]] <- huge_test_start$betas
test_names_extended[1] <- huge_test_start$test_name
test_ids_extended[1] <- huge_test_start$test_id

# Add first 150 original tests (positions 2-151)
for (i in 1:150) {
  maf1_list_extended[[i + 1]] <- maf1_list[[i]]
  maf2_list_extended[[i + 1]] <- maf2_list[[i]]
  beta_list_extended[[i + 1]] <- beta_list[[i]]
  test_names_extended[i + 1] <- test_names[i]
  test_ids_extended[i + 1] <- test_ids[i]
}

# Insert huge test at MIDDLE (position 152)
maf1_list_extended[[152]] <- huge_test_middle$maf1
maf2_list_extended[[152]] <- huge_test_middle$maf2
beta_list_extended[[152]] <- huge_test_middle$betas
test_names_extended[152] <- huge_test_middle$test_name
test_ids_extended[152] <- huge_test_middle$test_id

# Add remaining 150 original tests (positions 153-302)
for (i in 151:300) {
  maf1_list_extended[[i + 2]] <- maf1_list[[i]]
  maf2_list_extended[[i + 2]] <- maf2_list[[i]]
  beta_list_extended[[i + 2]] <- beta_list[[i]]
  test_names_extended[i + 2] <- test_names[i]
  test_ids_extended[i + 2] <- test_ids[i]
}

# Insert huge test at END (position 303)
maf1_list_extended[[303]] <- huge_test_end$maf1
maf2_list_extended[[303]] <- huge_test_end$maf2
beta_list_extended[[303]] <- huge_test_end$betas
test_names_extended[303] <- huge_test_end$test_name
test_ids_extended[303] <- huge_test_end$test_id

cat(sprintf("Strategic placement:\n"))
cat(sprintf("  Position 1: %s (START - batch 1)\n", test_names_extended[1]))
cat(sprintf("  Position 152: %s (MIDDLE - batch 3)\n", test_names_extended[152]))
cat(sprintf("  Position 303: %s (END - batch 6)\n\n", test_names_extended[303]))

cat(sprintf("Total tests: %d (original %d + 3 huge tests)\n\n", 
            length(maf1_list_extended), length(maf1_list)))


Adding 3 extremely large tests at START, MIDDLE, and END...
  Huge test START: Hemoglobin_concentration (continuous-6667-male) - 85000 SNPs
  Huge test MIDDLE: Vitamin_D (physical_measures-3656-female) - 55000 SNPs
  Huge test END: White_blood_cell_count (categorical-2118-male) - 35000 SNPs

Strategic placement:
  Position 1: Hemoglobin_concentration (START - batch 1)
  Position 152: Vitamin_D (MIDDLE - batch 3)
  Position 303: White_blood_cell_count (END - batch 6)

Total tests: 303 (original 300 + 3 huge tests)



In [17]:
# Run V10 Adaptive on EXTENDED dataset (303 tests with 3 huge ones)
cat("Running V10 Adaptive Sequential GPU on extended dataset (303 tests)...\n")
cat("Expected: 3 huge tests will be excluded due to OOM\n\n")

gpu_cleanup()  # Clean GPU memory first

t_seq_extended <- system.time({
  results_seq_extended <- lapply(1:length(maf1_list_extended), function(i) {
    if (i %% 50 == 0) cat(sprintf("  V10 Adaptive test %d/%d...\n", i, length(maf1_list_extended)))
    compute_Qx_sequential_gpu_pure_adaptive(
      maf1_list_extended[[i]], 
      maf2_list_extended[[i]], 
      beta_list_extended[[i]], 
      n_perm = n_perm, 
      seed = seed + i,
      test_name = test_names_extended[i],
      test_id = test_ids_extended[i]
    )
  })
})

# Count excluded tests
n_excluded_seq <- sum(sapply(results_seq_extended, function(x) isTRUE(x$excluded)))
n_successful_seq <- sum(sapply(results_seq_extended, function(x) !isTRUE(x$excluded)))

cat("\n")
cat(strrep("=", 80), "\n")
cat("V10 ADAPTIVE SUMMARY (Extended Dataset)\n")
cat(strrep("=", 80), "\n")
cat(sprintf("Total tests: %d\n", length(results_seq_extended)))
cat(sprintf("Successful: %d\n", n_successful_seq))
cat(sprintf("Excluded (OOM): %d\n", n_excluded_seq))
cat(sprintf("Time: %.2f seconds\n\n", t_seq_extended[3]))

# Show which tests were excluded
if (n_excluded_seq > 0) {
  cat("Excluded tests:\n")
  for (i in 1:length(results_seq_extended)) {
    if (isTRUE(results_seq_extended[[i]]$excluded)) {
      cat(sprintf("  Position %d: %s (%s) - %d SNPs\n", 
                  i, 
                  results_seq_extended[[i]]$test_name,
                  results_seq_extended[[i]]$test_id,
                  results_seq_extended[[i]]$n_snps))
    }
  }
  cat("\n")
}

Running V10 Adaptive Sequential GPU on extended dataset (303 tests)...
Expected: 3 huge tests will be excluded due to OOM

🔥 GPU MEMORY CLEANUP

Step 1: Synchronizing GPU...
Step 2: Freeing memory pools...
Step 3: CPU garbage collection...

✅ GPU memory cleaned successfully!
ℹ️  Memory should be fully released (check nvidia-smi)
  ⚠️  OUT OF MEMORY for test: Hemoglobin_concentration (continuous-6667-male) with 81638 SNPs
      Error: cupy.cuda.memory.OutOfMemoryError: Out of memory allocating 6,531,040,256 bytes (allocated so far: 29,423,604,736 bytes).
Run `reticulate::py_last_err
      Test EXCLUDED!

  V10 Adaptive test 50/303...
  V10 Adaptive test 100/303...
  V10 Adaptive test 150/303...
  ⚠️  OUT OF MEMORY for test: Vitamin_D (physical_measures-3656-female) with 52824 SNPs
      Error: cupy.cuda.memory.OutOfMemoryError: Out of memory allocating 4,225,920,000 bytes (allocated so far: 27,100,377,088 bytes).
Run `reticulate::py_last_err
      Test EXCLUDED!

  V10 Adaptive test 200

In [18]:
# Run V11 Batch GPU on EXTENDED dataset (303 tests with 3 huge ones)
cat("Running V11 Batch GPU on extended dataset (303 tests)...\n")


gpu_cleanup()  # Clean GPU memory first

t_batch_extended <- system.time({
  results_batch_extended_output <- compute_Qx_batch_gpu_adaptive(
    maf1_list_extended, 
    maf2_list_extended, 
    beta_list_extended,
    test_names = test_names_extended,
    test_ids = test_ids_extended,
    n_perm = n_perm, 
    seed = seed,
    batch_size = 7
  )
})

cat(sprintf("\n✅ V11 Batch completed in %.2f seconds\n", t_batch_extended[3]))
cat(sprintf("   Speedup vs V10 Adaptive: %.1fx\n\n", t_seq_extended[3] / t_batch_extended[3]))

Running V11 Batch GPU on extended dataset (303 tests)...
🔥 GPU MEMORY CLEANUP

Step 1: Synchronizing GPU...
Step 2: Freeing memory pools...
Step 3: CPU garbage collection...

✅ GPU memory cleaned successfully!
ℹ️  Memory should be fully released (check nvidia-smi)
VERSION 11: BATCH GPU WITH PER-TEST MATRICES

Starting with 303 tests, batch size: 7, permutations: 10000
Pre-filtering NAs from all tests...
No neutral SNPs provided — using per-locus F_ST from GWAS SNPs

--- Batch 1/44: Tests 1-7 ---
  Processing 7 tests (max SNPs: 81638)...
  OOM Error: cupy.cuda.memory.OutOfMemoryError: Out of memory allocating 6,531,040,256 bytes (allocated so far: 30,384,001,024 bytes).
Run `reticulate::py_last_error()` for details.
  Removing largest test: Hemoglobin_concentration (81638 SNPs)
  Retrying with 6 tests...
  Processing 6 tests (max SNPs: 534)...
  Transferring results from GPU...
  Batch completed successfully! (6 tests)

--- Batch 2/44: Tests 8-14 ---
  Processing 7 tests (max SNPs: 518)

In [19]:
cat(strrep("=", 80), "\n")
cat("EXTENDED DATASET VERIFICATION: V10 Adaptive vs V11 Batch\n")
cat(strrep("=", 80), "\n\n")

cat("Comparing Qx values for ALL successfully processed tests...\n\n")

# Get results lists
results_batch_extended <- results_batch_extended_output$results

# Track comparison statistics
all_match_extended <- TRUE
max_diff_extended <- 0
n_compared <- 0
n_both_excluded <- 0
n_v10_only_excluded <- 0
n_v11_only_excluded <- 0

mismatches <- list()

for (i in 1:length(results_seq_extended)) {
  seq_result <- results_seq_extended[[i]]
  batch_result <- results_batch_extended[[i]]
  
  # Check exclusion status
  seq_excluded <- isTRUE(seq_result$excluded) || is.null(batch_result)
  batch_excluded <- is.null(batch_result) || is.na(batch_result$Qx)
  
  if (seq_excluded && batch_excluded) {
    n_both_excluded <- n_both_excluded + 1
    next
  }
  
  if (seq_excluded && !batch_excluded) {
    n_v10_only_excluded <- n_v10_only_excluded + 1
    cat(sprintf("  ⚠️  Test %d excluded by V10 but processed by V11: %s\n", i, test_names_extended[i]))
    next
  }
  
  if (!seq_excluded && batch_excluded) {
    n_v11_only_excluded <- n_v11_only_excluded + 1
    cat(sprintf("  ⚠️  Test %d excluded by V11 but processed by V10: %s\n", i, test_names_extended[i]))
    next
  }
  
  # Both methods succeeded - compare Qx values
  n_compared <- n_compared + 1
  
  qx_seq <- seq_result$Qx
  qx_batch <- batch_result$Qx
  
  diff <- abs(qx_seq - qx_batch)
  max_diff_extended <- max(max_diff_extended, diff, na.rm = TRUE)
  
  # Check if they match (tolerance: 1e-5)
  if (diff >= 1e-5) {
    all_match_extended <- FALSE
    mismatches[[length(mismatches) + 1]] <- list(
      index = i,
      test_name = test_names_extended[i],
      qx_seq = qx_seq,
      qx_batch = qx_batch,
      diff = diff
    )
  }
}

cat("\n")
cat(strrep("-", 80), "\n")
cat("COMPARISON SUMMARY:\n")
cat(sprintf("  Total tests in extended dataset: %d\n", length(results_seq_extended)))
cat(sprintf("  Tests excluded by both methods: %d\n", n_both_excluded))
cat(sprintf("  Tests excluded only by V10: %d\n", n_v10_only_excluded))
cat(sprintf("  Tests excluded only by V11: %d\n", n_v11_only_excluded))
cat(sprintf("  Tests successfully compared: %d\n", n_compared))
cat(sprintf("  Max Qx difference: %.2e\n\n", max_diff_extended))

if (length(mismatches) > 0) {
  cat(sprintf("❌ MISMATCHES FOUND: %d tests differ by > 1e-5\n\n", length(mismatches)))
  cat("First few mismatches:\n")
  for (i in 1:min(3, length(mismatches))) {
    m <- mismatches[[i]]
    cat(sprintf("  Test %d: %s\n", m$index, m$test_name))
    cat(sprintf("    V10 Qx: %.10f\n", m$qx_seq))
    cat(sprintf("    V11 Qx: %.10f\n", m$qx_batch))
    cat(sprintf("    Diff:   %.2e\n\n", m$diff))
  }
} else {
  cat("✅ VERIFICATION PASSED!\n")
  cat("   All successfully processed tests match between V10 and V11\n")
  cat(sprintf("   (Max difference: %.2e < 1e-5 tolerance)\n\n", max_diff_extended))
}

# Show detailed breakdown of excluded tests
cat(strrep("-", 80), "\n")
cat("EXCLUSION DETAILS:\n\n")

if (n_both_excluded > 0) {
  cat(sprintf("Tests excluded by BOTH V10 and V11 (%d):\n", n_both_excluded))
  for (i in 1:length(results_seq_extended)) {
    seq_excluded <- isTRUE(results_seq_extended[[i]]$excluded) || is.null(results_batch_extended[[i]])
    batch_excluded <- is.null(results_batch_extended[[i]]) || is.na(results_batch_extended[[i]]$Qx)
    
    if (seq_excluded && batch_excluded) {
      cat(sprintf("  Position %d: %s - %d SNPs\n", 
                  i, 
                  test_names_extended[i],
                  length(beta_list_extended[[i]])))
    }
  }
  cat("\n")
}

cat(strrep("=", 80), "\n")
cat("FINAL SUMMARY\n")
cat(strrep("=", 80), "\n")
cat(sprintf("V10 Adaptive Sequential: %.2f sec, %d successful, %d excluded\n", 
            t_seq_extended[3], n_successful_seq, n_excluded_seq))
cat(sprintf("V11 Batch GPU:          %.2f sec, %d successful, %d excluded\n", 
            t_batch_extended[3], results_batch_extended_output$n_successful, results_batch_extended_output$n_excluded))
cat(sprintf("Speedup (V11 vs V10):   %.1fx\n", t_seq_extended[3] / t_batch_extended[3]))
cat(sprintf("Qx accuracy:            %s (max diff: %.2e)\n", 
            ifelse(all_match_extended, "PERFECT", "ISSUES FOUND"), max_diff_extended))

EXTENDED DATASET VERIFICATION: V10 Adaptive vs V11 Batch

Comparing Qx values for ALL successfully processed tests...


-------------------------------------------------------------------------------- 
COMPARISON SUMMARY:
  Total tests in extended dataset: 303
  Tests excluded by both methods: 2
  Tests excluded only by V10: 0
  Tests excluded only by V11: 0
  Tests successfully compared: 301
  Max Qx difference: 3.55e-15

✅ VERIFICATION PASSED!
   All successfully processed tests match between V10 and V11
   (Max difference: 3.55e-15 < 1e-5 tolerance)

-------------------------------------------------------------------------------- 
EXCLUSION DETAILS:

Tests excluded by BOTH V10 and V11 (2):
  Position 1: Hemoglobin_concentration - 85000 SNPs
  Position 152: Vitamin_D - 55000 SNPs

FINAL SUMMARY
V10 Adaptive Sequential: 218.72 sec, 301 successful, 2 excluded
V11 Batch GPU:          83.38 sec, 301 successful, 2 excluded
Speedup (V11 vs V10):   2.6x
Qx accuracy:            PERFECT (max 

## 8. Stress Test: When Does V11 Batch Really Shine? 🔥

The extended dataset showed minimal speedup because:
- Tests were too large (85K, 55K, 35K SNPs) → hit GPU memory limit
- Both methods excluded the same tests

**Let's create a better scenario for V11:**
- **Many medium-sized tests** (2000-5000 SNPs) - large enough to benefit from batching
- **Not too large** - won't hit memory limits
- **Uniform distribution** - V11 can batch efficiently without OOM issues

This simulates a real GWAS scenario with hundreds of moderately complex traits.

### Note on the batch size
 **it is extremely important to chose the batch size wisely:** a batch size too small does not take full advantage of parallelization;
 a batch size too big however is catastrophic in terms of performances as it causes out of memory errors at every batch and the function repeat the computation of the same batch possibly multiple times while removing the biggest element. Thus, **it is strongly raccomended to run some tests to correctly identify the best batch size your gpu memory allows before starting the full analysis**. In my case, i have noticed that a batch size of 7 is ok for my 5060 ti 16 gb. 


In [20]:
# Generate STRESS TEST dataset: 100 tests with medium-large SNP counts
# This is the sweet spot where V11 batching shows real advantage!

n_tests_stress <- 100
n_perm_stress <- 10000
seed_stress <- 456

cat("Generating STRESS TEST dataset:\n")
cat("  - 100 tests with 2000-5000 SNPs each\n")
cat("  - Large enough to show batching advantage\n")
cat("  - Small enough to avoid OOM issues\n\n")

test_data_stress <- vector("list", n_tests_stress)
set.seed(seed_stress)

for (i in 1:n_tests_stress) {
  # Medium-large SNP counts (2000-5000 range)
  # This is realistic for GWAS meta-analyses with filtered SNP sets
  n_snps <- sample(2000:5000, 1)
  pop1_maf <- runif(1, 0.25, 0.35)
  pop2_maf <- runif(1, 0.2, 0.3)
  
  test_data_stress[[i]] <- generate_simulated_data(
    n_snps = n_snps,
    pop1_mean_maf = pop1_maf,
    pop2_mean_maf = pop2_maf,
    na_rate = 0.02
  )
}

cat("✅ Generated stress test dataset\n\n")

# Prepare data lists
maf1_list_stress <- lapply(test_data_stress, function(x) x$maf1)
maf2_list_stress <- lapply(test_data_stress, function(x) x$maf2)
beta_list_stress <- lapply(test_data_stress, function(x) x$betas)
test_names_stress <- sapply(test_data_stress, function(x) x$test_name)
test_ids_stress <- sapply(test_data_stress, function(x) x$test_id)

# Show SNP distribution
snp_counts_stress <- sapply(beta_list_stress, length)
cat("SNP count distribution:\n")
cat(sprintf("  Min:    %0f SNPs\n", min(snp_counts_stress)))
cat(sprintf("  Median: %0f SNPs\n", median(snp_counts_stress)))
cat(sprintf("  Mean:   %.0f SNPs\n", mean(snp_counts_stress)))
cat(sprintf("  Max:    %0f SNPs\n\n", max(snp_counts_stress)))

Generating STRESS TEST dataset:
  - 100 tests with 2000-5000 SNPs each
  - Large enough to show batching advantage
  - Small enough to avoid OOM issues



✅ Generated stress test dataset

SNP count distribution:
  Min:    2017.000000 SNPs
  Median: 3539.000000 SNPs
  Mean:   3524 SNPs
  Max:    4986.000000 SNPs



In [21]:
# Run V10 Adaptive on STRESS TEST dataset
cat("Running V10 Adaptive Sequential GPU on stress test (100 tests, 2K-5K SNPs)...\n\n")

gpu_cleanup()  # Clean GPU memory first

t_seq_stress <- system.time({
  results_seq_stress <- lapply(1:n_tests_stress, function(i) {
    if (i %% 50 == 0) cat(sprintf("  V10 Adaptive test %d/%d...\n", i, n_tests_stress))
    compute_Qx_sequential_gpu_pure_adaptive(
      maf1_list_stress[[i]], 
      maf2_list_stress[[i]], 
      beta_list_stress[[i]], 
      n_perm = n_perm_stress, 
      seed = seed_stress + i,
      test_name = test_names_stress[i],
      test_id = test_ids_stress[i]
    )
  })
})

# Count results
n_excluded_seq_stress <- sum(sapply(results_seq_stress, function(x) isTRUE(x$excluded)))
n_successful_seq_stress <- sum(sapply(results_seq_stress, function(x) !isTRUE(x$excluded)))

cat("\n")
cat(sprintf("✅ V10 Adaptive completed in %.2f seconds\n", t_seq_stress[3]))
cat(sprintf("   Successful: %d, Excluded: %d\n\n", n_successful_seq_stress, n_excluded_seq_stress))

Running V10 Adaptive Sequential GPU on stress test (100 tests, 2K-5K SNPs)...

🔥 GPU MEMORY CLEANUP

Step 1: Synchronizing GPU...
Step 2: Freeing memory pools...
Step 3: CPU garbage collection...

✅ GPU memory cleaned successfully!
ℹ️  Memory should be fully released (check nvidia-smi)
  ⚠️  OUT OF MEMORY for test: Triglycerides (biomarkers-9961-both_sexes) with 2813 SNPs
      Error: cupy.cuda.memory.OutOfMemoryError: Out of memory allocating 225,040,384 bytes (allocated so far: 30,264,812,544 bytes).
Run `reticulate::py_last_error
      Test EXCLUDED!

  ⚠️  OUT OF MEMORY for test: Triglycerides (physical_measures-5637-both_sexes) with 3518 SNPs
      Error: cupy.cuda.memory.OutOfMemoryError: Out of memory allocating 281,440,256 bytes (allocated so far: 30,223,587,328 bytes).
Run `reticulate::py_last_error
      Test EXCLUDED!

  V10 Adaptive test 50/100...
  ⚠️  OUT OF MEMORY for test: Hair_colour (continuous-5464-both_sexes) with 2342 SNPs
      Error: cupy.cuda.memory.OutOfMemoryE

In [22]:
# Run V11 Batch GPU on STRESS TEST dataset
cat("Running V11 Batch GPU on stress test (100 tests, 2K-5K SNPs)...\n\n")

gpu_cleanup()  # Clean GPU memory first

t_batch_stress <- system.time({
  results_batch_stress_output <- compute_Qx_batch_gpu_adaptive(
    maf1_list_stress, 
    maf2_list_stress, 
    beta_list_stress,
    test_names = test_names_stress,
    test_ids = test_ids_stress,
    n_perm = n_perm_stress, 
    seed = seed_stress,
    batch_size = 7  # Larger batch size should work well here
  )
})

cat(sprintf("\n✅ V11 Batch completed in %.2f seconds\n", t_batch_stress[3]))
cat(sprintf("   Successful: %d, Excluded: %d\n\n", 
            results_batch_stress_output$n_successful, 
            results_batch_stress_output$n_excluded))

Running V11 Batch GPU on stress test (100 tests, 2K-5K SNPs)...

🔥 GPU MEMORY CLEANUP

Step 1: Synchronizing GPU...
Step 2: Freeing memory pools...
Step 3: CPU garbage collection...

✅ GPU memory cleaned successfully!
ℹ️  Memory should be fully released (check nvidia-smi)
VERSION 11: BATCH GPU WITH PER-TEST MATRICES

Starting with 100 tests, batch size: 7, permutations: 10000
Pre-filtering NAs from all tests...
No neutral SNPs provided — using per-locus F_ST from GWAS SNPs

--- Batch 1/15: Tests 1-7 ---
  Processing 7 tests (max SNPs: 4171)...
  Transferring results from GPU...
  Batch completed successfully! (7 tests)

--- Batch 2/15: Tests 8-14 ---
  Processing 7 tests (max SNPs: 4787)...
  Transferring results from GPU...
  Batch completed successfully! (7 tests)

--- Batch 3/15: Tests 15-21 ---
  Processing 7 tests (max SNPs: 4781)...
  Transferring results from GPU...
  Batch completed successfully! (7 tests)

--- Batch 4/15: Tests 22-28 ---
  Processing 7 tests (max SNPs: 4418)..

In [23]:
cat(strrep("=", 80), "\n")
cat("STRESS TEST RESULTS: V10 vs V11 Performance Comparison\n")
cat(strrep("=", 80), "\n\n")

# Get results
results_batch_stress <- results_batch_stress_output$results
num_tests_verified <- min(n_tests_stress, 50)
# Verify accuracy (spot check first 50 tests)
cat(sprintf("Verifying accuracy (first %d tests)...\n", num_tests_verified))
all_match_stress <- TRUE
max_diff_stress <- 0

for (i in 1:num_tests_verified) {
  seq_result <- results_seq_stress[[i]]
  batch_result <- results_batch_stress[[i]]
  
  if (!isTRUE(seq_result$excluded) && !is.null(batch_result)) {
    diff <- abs(seq_result$Qx - batch_result$Qx)
    max_diff_stress <- max(max_diff_stress, diff, na.rm = TRUE)
    
    if (diff >= 1e-5) {
      all_match_stress <- FALSE
      cat(sprintf("  ⚠️  Test %d differs by %.2e\n", i, diff))
    }
  }
}

if (all_match_stress) {
  cat(sprintf("✅ All spot checks passed (max diff: %.2e)\n\n", max_diff_stress))
} else {
  cat("❌ Some tests differ!\n\n")
}

# Performance comparison
cat(strrep("-", 80), "\n")
cat("PERFORMANCE COMPARISON:\n\n")

cat("Dataset characteristics:\n")
cat(sprintf("  Tests: %d\n", n_tests_stress))
cat(sprintf("  SNPs per test: %d - %d (avg: %.0f)\n", 
            min(snp_counts_stress), max(snp_counts_stress), mean(snp_counts_stress)))
cat(sprintf("  Permutations: %d\n\n", n_perm_stress))

cat("V10 Adaptive Sequential:\n")
cat(sprintf("  Time:       %.2f seconds\n", t_seq_stress[3]))
cat(sprintf("  Successful: %d tests\n", n_successful_seq_stress))
cat(sprintf("  Excluded:   %d tests\n\n", n_excluded_seq_stress))

cat("V11 Batch GPU:\n")
cat(sprintf("  Time:       %.2f seconds\n", t_batch_stress[3]))
cat(sprintf("  Successful: %d tests\n", results_batch_stress_output$n_successful))
cat(sprintf("  Excluded:   %d tests\n\n", results_batch_stress_output$n_excluded))

speedup_stress <- t_seq_stress[3] / t_batch_stress[3]

cat(strrep("-", 80), "\n")
cat("SPEEDUP ANALYSIS:\n")
cat(sprintf("  V11 is %.2fx FASTER than V10\n", speedup_stress))
cat(sprintf("  Time saved: %.2f seconds (%.1f%%)\n", 
            t_seq_stress[3] - t_batch_stress[3],
            (1 - 1/speedup_stress) * 100))

cat("\n")
cat(strrep("=", 80), "\n")
cat("CONCLUSION:\n")
cat(strrep("=", 80), "\n\n")

if (speedup_stress > 2) {
  cat(sprintf("🚀 V11 shows SIGNIFICANT speedup (%.1fx) with medium-large tests!\n", speedup_stress))
  cat("\n   This dataset is ideal for V11 batching because:\n")
  cat("   ✅ Tests are large enough to benefit from parallel processing\n")
  cat("   ✅ Tests fit comfortably in GPU memory (no OOM issues)\n")
  cat("   ✅ Uniform size distribution allows efficient batching\n\n")
  cat("   V10 processes one-by-one, synchronizing after each test.\n")
  cat("   V11 queues all operations and syncs once per batch.\n")
} else {
  cat(sprintf("V11 speedup is modest (%.1fx) - tests may still be at memory boundary\n", speedup_stress))
}

STRESS TEST RESULTS: V10 vs V11 Performance Comparison

Verifying accuracy (first 50 tests)...
✅ All spot checks passed (max diff: 1.78e-15)

-------------------------------------------------------------------------------- 
PERFORMANCE COMPARISON:

Dataset characteristics:
  Tests: 100
  SNPs per test: 2017 - 4986 (avg: 3524)
  Permutations: 10000

V10 Adaptive Sequential:
  Time:       582.91 seconds
  Successful: 95 tests
  Excluded:   5 tests

V11 Batch GPU:
  Time:       20.91 seconds
  Successful: 100 tests
  Excluded:   0 tests

-------------------------------------------------------------------------------- 
SPEEDUP ANALYSIS:
  V11 is 27.88x FASTER than V10
  Time saved: 562.00 seconds (96.4%)

CONCLUSION:

🚀 V11 shows SIGNIFICANT speedup (27.9x) with medium-large tests!

   This dataset is ideal for V11 batching because:
   ✅ Tests are large enough to benefit from parallel processing
   ✅ Tests fit comfortably in GPU memory (no OOM issues)
   ✅ Uniform size distribution allows 